In [1]:
# Imports
import time
import librosa
import numpy as np
import jiwer
import pandas as pd
from whisper.normalizers.basic import BasicTextNormalizer
from tqdm.notebook import tqdm
from qwen_asr import Qwen3ASRModel

In [2]:
# Cargar modelo con backend vLLM (el streaming solo funciona con vLLM)
repo_id = "Qwen/Qwen3-ASR-1.7B"
print(f"Cargando modelo: {repo_id}")

model = Qwen3ASRModel.LLM(
    model=repo_id,
    gpu_memory_utilization=0.88,
    max_model_len=24576,
    swap_space=2,
    max_new_tokens=4096,
)

Cargando modelo: Qwen/Qwen3-ASR-1.7B
INFO 09-19 17:22:35 [utils.py:263] non-default args: {'max_model_len': 24576, 'swap_space': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-ASR-1.7B'}
INFO 09-19 17:22:36 [model.py:530] Resolved architecture: Qwen3ASRForConditionalGeneration


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 09-19 17:22:37 [model.py:1545] Using max model len 24576
INFO 09-19 17:22:39 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-19 17:22:39 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 09-19 17:22:39 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


WARNING 09-19 17:22:40 [interface.py:470] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
WARNING 09-19 17:22:40 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=641) INFO 09-19 17:22:49 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='Qwen/Qwen3-ASR-1.7B', speculative_config=None, tokenizer='Qwen/Qwen3-ASR-1.7B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=24576, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, 

...

(EngineCore_DP0 pid=641) INFO 09-19 17:23:09 [cuda.py:351] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:31<00:31, 31.96s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:36<00:00, 16.04s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:36<00:00, 18.43s/it]
(EngineCore_DP0 pid=641) 


(EngineCore_DP0 pid=641) INFO 09-19 17:23:47 [default_loader.py:291] Loading weights took 36.87 seconds
(EngineCore_DP0 pid=641) INFO 09-19 17:23:49 [gpu_model_runner.py:3905] Model loading took 3.87 GiB memory and 47.743388 seconds
(EngineCore_DP0 pid=641) INFO 09-19 17:23:50 [gpu_model_runner.py:4715] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 21 audio items of the maximum feature size.
(EngineCore_DP0 pid=641) INFO 09-19 17:24:06 [backends.py:644] Using cache directory: /home/carru/.cache/vllm/torch_compile_cache/330f7064d2/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=641) INFO 09-19 17:24:06 [backends.py:704] Dynamo bytecode transform time: 11.99 s
(EngineCore_DP0 pid=641) INFO 09-19 17:24:17 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 5.534 s
(EngineCore_DP0 pid=641) INFO 09-19 17:24:17 [monitor.py:34] torch.compile takes 17.52 s in total
(EngineCore_DP0 pid=641) INF

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.73it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 15.72it/s]


(EngineCore_DP0 pid=641) INFO 09-19 17:24:26 [gpu_model_runner.py:4856] Graph capturing finished in 7 secs, took 0.45 GiB
(EngineCore_DP0 pid=641) INFO 09-19 17:24:26 [core.py:273] init engine (profile, create kv cache, warmup model) took 36.39 seconds
INFO 09-19 17:24:27 [llm.py:347] Supported tasks: ['generate', 'transcription']


In [3]:
N_AUDIOS = 20
sr = 16000
CHUNKS_SEC = [0.48, 1.0, 2.0, 4.0]

base = "../../Europarl-ST/es/en/test"
lst_path = base + "/Europarl-ST.v2.es.en.test.lst"

audios = []
with open(lst_path, "r", encoding="utf-8") as lista_audios:
    for linea in lista_audios:
        if linea.strip():
            audios.append(str(linea).strip())

audios = audios[:N_AUDIOS]

print(f"Audios: {len(audios)} | configuraciones: {CHUNKS_SEC}")

Audios: 20 | configuraciones: [0.48, 1.0, 2.0, 4.0]


In [4]:
def transcribir_streaming(audio_array, chunk_sec):
    estado = model.init_streaming_state(
        language="Spanish",
        chunk_size_sec=chunk_sec,
    )

    paso = int(chunk_sec * sr)
    tiempos = []

    t0 = time.perf_counter()
    for i in range(0, len(audio_array), paso):
        trozo = audio_array[i:i + paso]
        inicio = time.perf_counter()
        model.streaming_transcribe(trozo, estado)
        tiempos.append((time.perf_counter() - inicio) * 1000)

    model.finish_streaming_transcribe(estado)
    total = time.perf_counter() - t0

    return estado.text, np.array(tiempos), total

In [ ]:
filas = []

for chunk_sec in CHUNKS_SEC:
    for audio in tqdm(audios, desc=f"chunk {chunk_sec} s"):

        ref_path = base + "/%s/transcription.tok" % audio
        with open(ref_path, "r", encoding="utf-8") as referencia:
            ref = referencia.read()

        audio_path = base + "/%s/audio_clip_diarization.m4a" % audio
        audio_array, _ = librosa.load(audio_path, sr=sr)

        duracion = len(audio_array) / sr

        try:
            hyp, por_chunk, total = transcribir_streaming(audio_array, chunk_sec)
            fallo = ""
        except Exception as e:
            hyp, por_chunk, total, fallo = "", np.array([np.nan]), np.nan, type(e).__name__
            print(f"  fallo en {audio} ({duracion:.0f} s): {str(e)[:90]}")

        filas.append(dict(
            audio=audio,
            chunk_sec=chunk_sec,
            fallo=fallo,
            latencia_computacional_ms=float(np.nanmean(por_chunk)),
            latencia_p90_ms=float(np.nanpercentile(por_chunk, 90)),
            latencia_primer_tercio_ms=float(np.nanmean(por_chunk[:len(por_chunk) // 3 or 1])),
            latencia_ultimo_tercio_ms=float(np.nanmean(por_chunk[-(len(por_chunk) // 3 or 1):])),
            duracion_audio_s=duracion,
            tiempo_total_s=total,
            rtf=total / duracion,
            hypothesis=hyp,
            reference=ref,
        ))

data = pd.DataFrame(filas)
data.to_csv("qwen_latencia_raw_es.csv", index=False)
data.head()

chunk 0.48 s:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_571/2942762236.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  fallo en en.20081008.24.3-238 (124 s): The decoder prompt (length 24577) is longer than the maximum model length of 24576. Make s


/tmp/ipykernel_571/2942762236.py:26: RuntimeWarning: Mean of empty slice
  latencia_computacional_ms=float(np.nanmean(por_chunk)),
/home/carru/venvs/qwen/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1409: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/tmp/ipykernel_571/2942762236.py:28: RuntimeWarning: Mean of empty slice
  latencia_primer_tercio_ms=float(np.nanmean(por_chunk[:len(por_chunk) // 3 or 1])),
/tmp/ipykernel_571/2942762236.py:29: RuntimeWarning: Mean of empty slice
  latencia_ultimo_tercio_ms=float(np.nanmean(por_chunk[-(len(por_chunk) // 3 or 1):])),
/tmp/ipykernel_571/2942762236.py:11: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_array, _ = librosa.load(audio_path, sr=sr)
/home/carru/venvs/qwen/lib/python3.12/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __aud

In [ ]:
# Normalización de texto para evaluación
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(str(t)) for t in data["hypothesis"]]
data["reference_clean"] = [normalizer(str(t)) for t in data["reference"]]

data.to_csv("qwen_latencia_limpio_es.csv", index=False)
data.head()

In [ ]:
# WER y latencia por configuración
resumen = []

for chunk_sec, grupo in data.groupby("chunk_sec"):
    ok = grupo[grupo["fallo"] == ""]
    wer = jiwer.wer(list(ok["reference_clean"]), list(ok["hypothesis_clean"]))
    computacional = ok["latencia_computacional_ms"].mean()
    resumen.append(dict(
        modelo="Qwen3-ASR",
        fallos=int((grupo["fallo"] != "").sum()),
        latencia_algoritmica_ms=chunk_sec * 1000,
        latencia_computacional_ms=computacional,
        latencia_total_ms=chunk_sec * 1000 + computacional,
        crecimiento=ok["latencia_ultimo_tercio_ms"].mean() / ok["latencia_primer_tercio_ms"].mean(),
        rtf=ok["rtf"].mean(),
        wer=wer * 100,
    ))

resumen = pd.DataFrame(resumen)
resumen.to_csv("qwen_latencia_resumen_es.csv", index=False)
resumen

In [ ]:
print("Si la latencia computacional supera el tamaño del chunk el sistema acumula retraso.")
print("El crecimiento compara el ultimo tercio del audio con el primero.")
print()

for _, f in resumen.iterrows():
    ms_chunk = f["latencia_algoritmica_ms"]
    sigue = "si" if f["latencia_computacional_ms"] < ms_chunk else "NO"
    print(
        f"chunk {ms_chunk:7.0f} ms | "
        f"computacional {f['latencia_computacional_ms']:7.1f} ms | "
        f"total {f['latencia_total_ms']:7.1f} ms | "
        f"crecimiento x{f['crecimiento']:.2f} | "
        f"WER {f['wer']:5.2f} % | tiempo real: {sigue}"
    )

*(Completar tras la ejecución: latencia y WER de Qwen3-ASR en streaming.)*